# Baselines - Part 4: training recipe comparison on B2 (fold I)

With TESSERA 2019 alone on fold I, the Method 2 setup reaches 62.69 test mIoU compared to 59.05 for B2. Both use the same evaluation protocol, while the training setups differ in learning rate (`1e-3` constant vs `1e-4` with cosine decay to 0) and class weighting (unweighted vs `1 / sqrt(frequency)`). Here we train B2 under four `lr=1e-3` configurations (seed 0, `32-64-128` UNet, 3,600 steps) and compare them against the existing B2 run on the fold I validation patches.

| Config | Run | Schedule | Class weights |
|---|---|---|---|
| B2 (Part 3) | `base_unet-t19-w32` | `lr=1e-4`, cosine | `1 / sqrt(frequency)` |
| `r1_lr1e3_cosine_sqrt` | `base_unet-t19-lr1e3-w32` | `lr=1e-3`, cosine | `1 / sqrt(frequency)` |
| `r2_lr1e3_cosine_none` | `base_unet-t19-lr1e3-nocw-w32` | `lr=1e-3`, cosine | none |
| `r3_lr1e3_constant_sqrt` | `base_unet-t19-lr1e3-const-w32` | `lr=1e-3`, constant | `1 / sqrt(frequency)` |
| `r4_lr1e3_constant_none` | `base_unet-t19-lr1e3-const-nocw-w32` | `lr=1e-3`, constant | none |

## 1. Environment and paths

In [ ]:
import json, os, shutil, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, f"| {os.cpu_count()} CPUs")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"  # or the branch being tested
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
RUNS, RUNS_CSV = DRIVE / "runs", DRIVE / "results" / "runs.csv"
OUT = DRIVE / "results" / "baselines"
RECIPES = {  # label: (config, run variant)
    "B2   lr 1e-4, cosine, 1/sqrt weights": ("configs/baselines/b2_unet_t19.yaml", "base_unet-t19-w32"),
    "r1   lr 1e-3, cosine, 1/sqrt weights": ("configs/recipe/r1_lr1e3_cosine_sqrt.yaml", "base_unet-t19-lr1e3-w32"),
    "r2   lr 1e-3, cosine, no weights": ("configs/recipe/r2_lr1e3_cosine_none.yaml", "base_unet-t19-lr1e3-nocw-w32"),
    "r3   lr 1e-3, constant, 1/sqrt weights": ("configs/recipe/r3_lr1e3_constant_sqrt.yaml", "base_unet-t19-lr1e3-const-w32"),
    "r4   lr 1e-3, constant, no weights": ("configs/recipe/r4_lr1e3_constant_none.yaml", "base_unet-t19-lr1e3-const-nocw-w32"),
}
assert (RUNS / "base_unet-t19-w32_100_foldI_s0" / "run.json").exists(), "B2 seed 0 missing: run notebooks/baselines/03_b2.ipynb first"

## 2. Repository and unit tests

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests
sys.path.insert(0, str(CODE))

## 3. Copy TESSERA 2019 data to local disk

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Train the four recipes (seed 0)
Runs `r1` through `r4` with seed 0 (`base_unet-t19-w32` from Part 3 is reused from Drive).

In [ ]:
for label, (cfg, _) in list(RECIPES.items())[1:]:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds 0

## 5. Validation comparison
Compares best validation mIoU, the step at which it was reached, and final validation mIoU across all five configurations, and plots the validation curves over the 3,600 training steps.

In [ ]:
import matplotlib.pyplot as plt

rows, fig = [], plt.figure(figsize=(9, 4.5))
for label, (_, v) in RECIPES.items():
    info = json.loads((RUNS / f"{v}_100_foldI_s0" / "run.json").read_text())
    hist = pd.DataFrame(info["history"])
    rows.append({"recipe": label, "best_val_miou": info["best_val_miou"], "best_step": info["best_step"],
                 "val_miou_end": hist.val_miou.iloc[-1], "minutes": info["train_seconds"] / 60})
    plt.plot(hist.step, hist.val_miou, label=label)
table = pd.DataFrame(rows).sort_values("best_val_miou", ascending=False)
pd.set_option("display.width", 220)
print(table.round(4).to_string(index=False))
plt.xlabel("step"); plt.ylabel("validation mIoU"); plt.title("B2 recipes, fold I, seed 0")
plt.grid(alpha=0.3); plt.legend(fontsize=8)
OUT.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT / "recipe_check_val_curves.png", dpi=130, bbox_inches="tight")
table.to_csv(OUT / "recipe_check_val.csv", index=False)
plt.show()